In [24]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("RetailPulseSilverPipeline")
    .config("spark.sql.warehouse.dir", "hdfs://namenode:8020/user/hive/warehouse")
    .config("hive.metastore.uris", "thrift://hive-metastore:9083")
    .enableHiveSupport()
    .getOrCreate()
)

In [36]:
bronze_df.printSchema()
bronze_df.select("ingestion_date", "event_timestamp").show(5, truncate=False)

root
 |-- event_id: string (nullable = true)
 |-- event_type: string (nullable = true)
 |-- event_timestamp: string (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- session_id: string (nullable = true)
 |-- order_id: integer (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- store_id: integer (nullable = true)
 |-- channel: string (nullable = true)
 |-- sequence: integer (nullable = true)
 |-- raw_payload: string (nullable = true)
 |-- kafka_ingest_ts: timestamp (nullable = true)
 |-- ingestion_date: date (nullable = true)

+--------------+--------------------------------+
|ingestion_date|event_timestamp                 |
+--------------+--------------------------------+
|2026-09-23    |event_timestamp                 |
|2026-09-23    |2026-09-23T08:13:46.860582+00:00|
|2026-09-23    |2026-09-23T07:56:20.860582+00:00|
|2026-09-23    |2026-09-23T08:47:27.860582+00:00|
|2026-09-23    |2026-09-23T08:18:31.860582+00:00|
+--------------+---------------------

In [38]:
from pyspark.sql.functions import (
    col, to_timestamp, lit, when, coalesce, trim, lower,
    row_number, date_format, unix_timestamp, abs as spark_abs
)
from pyspark.sql.window import Window

bronze_df = spark.table("bronze.retailpulse_events")
total_bronze = bronze_df.count()
print(f"Total bronze rows: {total_bronze}")

Total bronze rows: 10003


In [39]:
required_cols = ["customer_id", "product_id", "store_id", "channel", "session_id", "sequence"]

checked_df = (
    bronze_df
    .withColumn("is_malformed", col("event_id").isNull() | col("event_type").isNull())
    .withColumn("event_ts_parsed", to_timestamp(col("event_timestamp")))
    .withColumn("is_invalid_timestamp", col("event_ts_parsed").isNull())
    .withColumn(
        "is_missing_required_field",
        col("customer_id").isNull() | col("product_id").isNull() | col("store_id").isNull()
        | col("session_id").isNull() | col("channel").isNull() | col("sequence").isNull()
    )
)

In [40]:
dedup_window = Window.partitionBy("event_id").orderBy(
    col("ingestion_date").asc(),
    col("event_ts_parsed").asc(),
    col("sequence").asc()
)

checked_df = checked_df.withColumn(
    "row_num",
    when(col("is_malformed"), lit(1)).otherwise(row_number().over(dedup_window))
).withColumn(
    "is_duplicate",
    (~col("is_malformed")) & (col("row_num") > 1)
)

In [41]:
rejection_reason = (
    when(col("is_malformed"), "malformed_json")
    .when(col("is_invalid_timestamp"), "invalid_timestamp")
    .when(col("is_missing_required_field"), "missing_required_field")
    .when(col("is_duplicate"), "duplicate_event_id")
    .otherwise(None)
)

checked_df = checked_df.withColumn("rejection_reason", rejection_reason)

In [42]:
passed_df = checked_df.filter(col("rejection_reason").isNull())

standardized_df = (
    passed_df
    .withColumn("channel", lower(trim(col("channel"))))
    .withColumn("event_type", lower(trim(col("event_type"))))
    .withColumn("session_id", trim(col("session_id")))
    .withColumn("event_timestamp", date_format(col("event_ts_parsed"), "yyyy-MM-dd HH:mm:ss.SSSSSS"))
    .withColumn("kafka_ingest_ts", date_format(col("kafka_ingest_ts"), "yyyy-MM-dd HH:mm:ss.SSSSSS"))
    .withColumn(
        "is_late_arrival",
        when(
            spark_abs(unix_timestamp(col("kafka_ingest_ts")) - unix_timestamp(col("event_ts_parsed"))) > 86400,
            True
        ).otherwise(False)
    )
)

silver_final_df = standardized_df.select(
    "event_id", "event_type", "event_timestamp", "customer_id",
    "session_id", "order_id", "product_id", "store_id", "channel",
    "sequence", "is_late_arrival", "kafka_ingest_ts", "ingestion_date"
)

In [43]:
quarantine_df = (
    checked_df
    .filter(col("rejection_reason").isNotNull())
    .select("event_id", "event_type", "raw_payload", "rejection_reason", "kafka_ingest_ts", "ingestion_date")
)

In [44]:
spark.sql("DROP TABLE IF EXISTS silver.retailpulse_events")
spark.sql("DROP TABLE IF EXISTS silver.retailpulse_events_rejected")

DataFrame[]

In [45]:
silver_path = "hdfs://namenode:8020/user/hadoop/silver/retailpulse_events"
quarantine_path = "hdfs://namenode:8020/user/hadoop/silver/retailpulse_events_rejected"

spark.sql("CREATE DATABASE IF NOT EXISTS silver")

spark.sql(f"""
CREATE EXTERNAL TABLE silver.retailpulse_events (
    event_id STRING, event_type STRING, event_timestamp TIMESTAMP,
    customer_id INT, session_id STRING, order_id INT, product_id INT,
    store_id INT, channel STRING, sequence INT, is_late_arrival BOOLEAN,
    kafka_ingest_ts TIMESTAMP
)
PARTITIONED BY (ingestion_date DATE)
ROW FORMAT DELIMITED FIELDS TERMINATED BY ',' LINES TERMINATED BY '\\n'
STORED AS TEXTFILE LOCATION '{silver_path}'
""")

spark.sql(f"""
CREATE EXTERNAL TABLE silver.retailpulse_events_rejected (
    event_id STRING, event_type STRING, raw_payload STRING,
    rejection_reason STRING, kafka_ingest_ts STRING
)
PARTITIONED BY (ingestion_date DATE)
ROW FORMAT DELIMITED FIELDS TERMINATED BY '\\t' LINES TERMINATED BY '\\n'
STORED AS TEXTFILE LOCATION '{quarantine_path}'
""")


(
    silver_final_df
    .repartition(1, "ingestion_date")
    .write.mode("overwrite")
    .partitionBy("ingestion_date")
    .csv(silver_path)
)

(
    quarantine_df
    .repartition(1, "ingestion_date")
    .write.mode("overwrite")
    .partitionBy("ingestion_date")
    .option("sep", "\t")
    .csv(quarantine_path)
)

spark.sql("MSCK REPAIR TABLE silver.retailpulse_events")
spark.sql("MSCK REPAIR TABLE silver.retailpulse_events_rejected")

DataFrame[]

In [46]:
metrics_df = (
    checked_df
    .groupBy(coalesce(col("rejection_reason"), lit("accepted")).alias("outcome"))
    .count()
    .withColumn("pct_of_total", (col("count") / total_bronze * 100))
)
metrics_df.show(truncate=False)

+------------------+-----+-----------------+
|outcome           |count|pct_of_total     |
+------------------+-----+-----------------+
|invalid_timestamp |98   |0.979706088173548|
|duplicate_event_id|496  |4.95851244626612 |
|accepted          |9409 |94.06178146556033|
+------------------+-----+-----------------+



In [47]:
spark.sql("SELECT * FROM silver.retailpulse_events LIMIT 10").show()
spark.sql("SELECT * FROM silver.retailpulse_events_rejected LIMIT 10").show()

+--------------------+----------------+--------------------+-----------+-------------+--------+----------+--------+-------+--------+---------------+---------------+--------------+
|            event_id|      event_type|     event_timestamp|customer_id|   session_id|order_id|product_id|store_id|channel|sequence|is_late_arrival|kafka_ingest_ts|ingestion_date|
+--------------------+----------------+--------------------+-----------+-------------+--------+----------+--------+-------+--------+---------------+---------------+--------------+
|0d1a4bd9-2970-48e...|    cart_updated|2026-09-23 07:42:...|      21459|session-29679|    null|      7908|      33|    web|    2776|          false|           null|    2026-09-23|
|16954f7e-48b3-47a...|  product_viewed|2026-09-23 07:53:...|      38832|session-73424|    null|      3637|      17| mobile|    3479|          false|           null|    2026-09-23|
|1707621c-91c5-4a6...|checkout_started|2026-09-23 07:38:...|      33961|session-60680|    null|     